In [ ]:
# ==============================
# 1ï¸âƒ£ Import Required Libraries
# ==============================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
# ==============================
# 2ï¸âƒ£ Define Dataset Paths
# ==============================

# Update these paths to your dataset directories
train_path = r'C:\Users\hp\Desktop\Tuberculosis and Pneumonia\main Dataset\split_data\train'
valid_path = r'C:\Users\hp\Desktop\Tuberculosis and Pneumonia\main Dataset\split_data\validation'
test_path  = r'C:\Users\hp\Desktop\Tuberculosis and Pneumonia\main Dataset\split_data\test'

BATCH_SIZE = 16  # Recommended for medical dataset
IMG_SIZE = (224, 224)

In [ ]:
# ==============================
# 3ï¸âƒ£ Data Preprocessing
# ==============================

# VGG16 requires its own preprocessing function
def tl_preprocess(x):
    return preprocess_input(x)

# Create ImageDataGenerators for train, validation, and test
tl_train_datagen = ImageDataGenerator(
    preprocessing_function=tl_preprocess
)

tl_valid_datagen = ImageDataGenerator(
    preprocessing_function=tl_preprocess
)

tl_test_datagen = ImageDataGenerator(
    preprocessing_function=tl_preprocess
)

# Load images from directories
tl_train_generator = tl_train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

tl_valid_generator = tl_valid_datagen.flow_from_directory(
    valid_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

tl_test_generator = tl_test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# ==============================
# 4ï¸âƒ£ Load Pretrained VGG16
# ==============================

# Load VGG16 pretrained on ImageNet
base_model = VGG16(
    weights='imagenet',       # Use pretrained ImageNet weights
    include_top=False,        # Remove original classification head
    input_shape=(224,224,3)   # Input shape must match dataset
)

# Freeze the base model for feature extraction phase
base_model.trainable = False


In [ ]:
# ==============================
# 5ï¸âƒ£ Build Custom Classification Head
# ==============================

model = keras.Sequential([
    
    base_model,                              # Pretrained feature extractor
    
    keras.layers.GlobalAveragePooling2D(),   # Convert feature maps to vector
    
    keras.layers.BatchNormalization(),       # Stabilize training
    
    keras.layers.Dense(256, activation='relu'),  # Fully connected layer
    
    keras.layers.Dropout(0.5),               # Reduce overfitting
    
    keras.layers.Dense(4, activation='softmax')  # Output layer (4 classes)
])


In [ ]:
# ==============================
# 6ï¸âƒ£ Compile Model (Feature Extraction Phase)
# ==============================

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # Small learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ==============================
# 7ï¸âƒ£ Callbacks (Professional Setup)
# ==============================

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'best_vgg16_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    verbose=1
)

callbacks = [early_stopping, model_checkpoint, reduce_lr]


In [ ]:
# ==============================
# 8ï¸âƒ£ Train (Feature Extraction)
# ==============================

history = model.fit(
    tl_train_generator,
    epochs=20,                    # EarlyStopping will stop automatically
    validation_data=tl_valid_generator,
    callbacks=callbacks
)

In [ ]:
# ==============================
# 9ï¸âƒ£ Fine-Tuning Phase
# ==============================

# Unfreeze last 20 layers of VGG16
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train again (fine-tuning)
fine_tune_history = model.fit(
    tl_train_generator,
    epochs=15,
    validation_data=tl_valid_generator,
    callbacks=callbacks
)

In [ ]:
# ==============================
# ðŸ”Ÿ Evaluate Model
# ==============================

test_loss, test_accuracy = model.evaluate(tl_test_generator)

print("Test Accuracy:", test_accuracy)

In [ ]:
# ==============================
# Confusion Matrix & Classification Report
# ==============================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions
Y_pred = model.predict(tl_test_generator)
y_pred = np.argmax(Y_pred, axis=1)

# True labels
y_true = tl_test_generator.classes

# Class labels
class_labels = list(tl_test_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_labels,
            yticklabels=class_labels,
            cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Classification Report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# ==============================
# Plot Training History
# ==============================

def plot_history(history):
    
    # Accuracy plot
    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'])
    plt.show()
    
    # Loss plot
    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'])
    plt.show()

plot_history(history)

In [ ]:
# ==============================
# Functional Model (Grad-CAM Safe)
# ==============================

from tensorflow.keras import layers, Model

# Base model
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# Define input explicitly
inputs = layers.Input(shape=(224,224,3))

# Forward pass through VGG16
x = base_model(inputs, training=False)

# Custom head
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(4, activation='softmax')(x)

# Create model
model = Model(inputs, outputs)

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):

    # Get VGG16 layer
    last_conv_layer = model.get_layer("vgg16").get_layer(last_conv_layer_name)

    # Create grad model
    grad_model = Model(
        [model.input],
        [last_conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
img_path = r"C:\Users\hp\Desktop\Tuberculosis and Pneumonia\main Dataset\split_data\test\Covid\00f9707e0de334db70a7924a9de98368.jpg"

img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224,224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

heatmap = make_gradcam_heatmap(img_array, model, "block5_conv3")

plt.matshow(heatmap)
plt.title("Grad-CAM Heatmap")
plt.colorbar()
plt.show()

In [ ]:
for layer in model.layers:
    print(layer.name)